# Chapter 2 - Working with Text

## Byte pair encoding

Byte pair encoding (BPE) was used to train LLMs such as GPT-2, GPT-3 and the
models used in ChatGPT.

In [8]:
from importlib.metadata import version
import tiktoken
print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.14.0


In [9]:
tokenizer = tiktoken.get_encoding("gpt2")

text = ("Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.")
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]


In [10]:
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


In [11]:
text = "Akwirw ier"
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

strings = tokenizer.decode(integers)
print(strings)

[33901, 86, 343, 86, 220, 959]
Akwirw ier


## Data Sampling with a sliding window

To train the LLM, we need to provide it with a series of words and then the target
word that it should predict, then we include that word in the series and take the
next word. This is called a sliding window.

To do this efficiently, we use PyTorch's built in `Dataset` and `DataLoader` classes
to load the data into tensors.

In [12]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
  def __init__(self, txt, tokenizer, max_length, stride) -> None:
    self.input_ids = []
    self.target_ids = []

    token_ids = tokenizer.encode(txt)

    # Use a sliding window to chunk the book into overlapping sequences of max_length
    for i in range(0, len(token_ids) - max_length, stride):
      input_chunk = token_ids[i:i + max_length]
      target_chunk = token_ids[i + 1: i + max_length + 1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  # Total number of rows in the dataset
  def __len__(self):
      return len(self.input_ids)

  # Returns a single row from the dataset
  def __getitem__(self, idx):
      return self.input_ids[idx], self.target_ids[idx]

Now we'll use the `GPTDatasetV1` to load the inputs in batches via a PyTorch `DataLoader`.

In [13]:
# A data loader to generate batches with input-width pairs
def create_dataloader_v1(txt:str, batch_size=4, max_length=256, stride=128,
                         shuffle=True, drop_last=True, num_workers=0) -> DataLoader:
  tokenizer = tiktoken.get_encoding("gpt2")
  dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
  dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=shuffle,
    # True drops the last batch if it is shorter than the specified batch_size
    # to prevent loss spikes during training
    drop_last=drop_last,
    # The number of CPU processes to use for preprocessing
    num_workers=num_workers
  )

  return dataloader

In [14]:
with open("../../ch02/01_main-chapter-code/the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()

dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=8, stride=2, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464, 1807, 3619,  402,  271]]), tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899]])]


We used `max_length=4` to make it easy to read. When training LLMs, it is common
to use 256 or more. If we pull the next batch, we can see that `stride=1` causes 
the window to shift by one token/position.

In [15]:
second_batch = next(data_iter)
print(first_batch)
print(second_batch)

[tensor([[  40,  367, 2885, 1464, 1807, 3619,  402,  271]]), tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899]])]
[tensor([[ 2885,  1464,  1807,  3619,   402,   271, 10899,  2138]]), tensor([[ 1464,  1807,  3619,   402,   271, 10899,  2138,   257]])]


The `batch_size=1` is also useful for illustration and uses less memory during 
training but can lead to more noisy model updates. It is a tradeoff and a 
hyperparameter that you want to experiment with during training.

Let's look at a larger batch size,

In [16]:


dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## Creating Token Embeddings

For simplicity, we'll use a vocabulary of 6 words and an embedding size of 3.

In [17]:
import torch

vocab_size = 6
output_dim = 3

# Use a set seed to create a predictible random embedding
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


Next we'll apply this embedding layer to one token to get the embedding vector.

In [18]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


Notice that this is the fourth row of the `embedding_layer`.

Next, we'll apply that to all four input IDs.

In [19]:
input_ids = torch.tensor([2,3,5,1])
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


Now that we've created embedding vectors from the token IDs, we'll add in
information about the token position within the text.

## Encoding word positions

Word positions can be encoded as relative or absolute. ChatGPT uses absolute that
is encoded as a part of the model training itself. Here we will use positional.
We'll use more realistic (but still small) `output_dim` and the full size of our
BPE tokenizer vocab.

In [20]:
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

Now we'll instantiate the data loader

In [22]:
max_length = 4
dataloader = create_dataloader_v1(
  raw_text, batch_size=8, max_length=max_length,
  stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token IDs:\n", inputs)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])


Now we use the embedding layer to embed these token IDs into 256-dimensional vectors.

In [23]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


For a GPT model's absolute embedding approach, we create another embedding
layer that has the same embedding dimensions

In [24]:
context_length = max_length
pos_embeddings_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embeddings_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


We now add these positional embeddings to the token embeddings.

In [25]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])
